# MLP 학습·평가 재구현

학습자의 요청으로 AI가 데이터 준비와 직접 연결 MLP 정의만 구성했다.
이 준비 코드는 무보조 구현 범위에 포함하지 않는다.
학습·평가 부분은 학습자가 기존 구현을 보지 않고 작성한다.

위에서부터 준비 코드 셀 4개를 실행한 다음, 아래 빈 셀에서 시작한다.
이 노트북은 `main.ipynb`의 커널 변수나 학습된 가중치에 의존하지 않는다.
데이터는 실행 시 Karpathy의 공개 `names.txt`를 가져온다.
CUDA를 사용할 수 있으면 CUDA, 아니면 CPU를 사용하며 실제 장치를 출력한다.

모델 참고: [Bengio et al. (2003), 2절](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf#page=7)
데이터 출처: [karpathy/makemore](https://github.com/karpathy/makemore/blob/master/names.txt)


In [1]:
import random
from urllib.request import urlopen

import torch
import torch.nn.functional as F

with urlopen("https://raw.githubusercontent.com/karpathy/makemore/master/names.txt", timeout=30) as response:
    words = response.read().decode("utf-8").splitlines()

chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
block_size = 3
embedding_dim = 10
hidden_size = 200
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("words:", len(words), "vocabulary:", vocab_size, "device:", device)

words: 32033 vocabulary: 27 device: cuda


In [2]:
def build_dataset(word_list):
    X, Y = [], []
    for word in word_list:
        context = [0] * block_size
        for ch in word + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return (
        torch.tensor(X, dtype=torch.long, device=device),
        torch.tensor(Y, dtype=torch.long, device=device),
    )

shuffled_words = words.copy()
random.Random(42).shuffle(shuffled_words)
n = len(shuffled_words)
train_end, dev_end = n * 8 // 10, n * 9 // 10
Xtr, Ytr = build_dataset(shuffled_words[:train_end])
Xdev, Ydev = build_dataset(shuffled_words[train_end:dev_end])
Xte, Yte = build_dataset(shuffled_words[dev_end:])
for split_name, split_X, split_Y in [("train", Xtr, Ytr), ("dev", Xdev, Ydev), ("test", Xte, Yte)]:
    print(split_name, split_X.shape, split_Y.shape)
del split_name, split_X, split_Y

train torch.Size([182625, 3]) torch.Size([182625])
dev torch.Size([22655, 3]) torch.Size([22655])
test torch.Size([22866, 3]) torch.Size([22866])


In [7]:
# 재실행하면 가중치와 난수 시퀀스를 처음 상태로 되돌립니다.
g_model = torch.Generator(device=device).manual_seed(2147483647)
g_batch = torch.Generator(device=device).manual_seed(42)
C = torch.randn((vocab_size, embedding_dim), generator=g_model, device=device, dtype=torch.float32)
W1 = torch.randn((block_size * embedding_dim, hidden_size), generator=g_model, device=device, dtype=torch.float32)
b1 = torch.randn(hidden_size, generator=g_model, device=device, dtype=torch.float32)
W2 = torch.randn((hidden_size, vocab_size), generator=g_model, device=device, dtype=torch.float32) * 0.02
b2 = torch.zeros(vocab_size, device=device, dtype=torch.float32)
W_direct = torch.zeros((block_size * embedding_dim, vocab_size), dtype=W1.dtype, device=W1.device)
parameters_direct = [C, W1, b1, W2, b2, W_direct]
for p in parameters_direct:
    p.requires_grad_(True)

print("parameters:", sum(p.numel() for p in parameters_direct))


parameters: 12707


In [8]:
def forward_direct(X):
    emb = C[X]
    flat = emb.flatten(start_dim=1)
    direct_scores = flat @ W_direct
    h = torch.tanh(flat @ W1 + b1)
    return h @ W2 + b2 + direct_scores

## 직접 작성할 부분

학습 함수와 평가 함수를 아래 빈 셀에서 직접 작성한다. 함수 구조와 이름은 직접 정한다.
작성한 코드를 검토하기 전에는 실제 학습을 호출하지 않는다.

준비된 객체:

| 이름 | 용도 |
|---|---|
| `Xtr`, `Ytr` | 학습 입력과 정답 |
| `Xdev`, `Ydev` | 검증 입력과 정답 |
| `Xte`, `Yte` | 최종 평가용 데이터. 이번 구현·튜닝에는 사용하지 않음 |
| `forward_direct` | 입력에서 logits를 계산하는 직접 연결 MLP |
| `parameters_direct` | 모델의 학습 가능한 가중치 목록 |
| `batch_size` | 기존 실험의 배치 크기 |
| `g_batch` | 배치 선택에 사용할 난수 생성기 |
| `device` | 현재 실행 장치 |

기존 비교 실험은 학습률 0.03, 20,000회였으며 여기서는 아직 실행하지 않았다.


In [9]:
def train_step_direct(iter, lr):
    for step in range(iter):
        indices = torch.randint(high=len(Xtr), size=(batch_size,), generator=g_batch, device="cuda")
        loss = F.cross_entropy(forward_direct(Xtr[indices]), Ytr[indices])

        for p in parameters_direct:
            p.grad = None

        loss.backward()

        with torch.no_grad():
            for p in parameters_direct:
                p -= p.grad * lr

@torch.no_grad()
def eval_direct(X, Y):
    return F.cross_entropy(forward_direct(X), Y)

In [10]:
train_loss = eval_direct(Xtr, Ytr).item()
dev_loss = eval_direct(Xdev, Ydev).item()

print(f"train_loss: {train_loss}")
print(f"dev_loss: {dev_loss}")

train_loss: 3.348036527633667
dev_loss: 3.348778486251831


In [11]:
lr = 0.03
iter = 1000

train_step_direct(iter, lr)

In [12]:
train_loss = eval_direct(Xtr, Ytr).item()
dev_loss = eval_direct(Xdev, Ydev).item()

print(f"train_loss: {train_loss}")
print(f"dev_loss: {dev_loss}")

train_loss: 2.4138410091400146
dev_loss: 2.4144763946533203
